# 01 — Real-time Inference Demo

This notebook demonstrates the Phoenix ML inference pipeline:
- Health check
- Single prediction
- Batch predictions with latency measurement
- Model info retrieval

**Prerequisites**: Run `docker compose up -d` and `docker compose -f docker-compose.airflow.yaml up -d`

In [ ]:
import random
import statistics
import time

import httpx

API_URL = "http://localhost:8001"
MODEL_ID = "credit-risk"
NUM_FEATURES = 30  # credit risk ONNX model expects 30 inputs

## 1. Health Check

In [ ]:
resp = httpx.get(f"{API_URL}/health")
print(f"Status: {resp.status_code}")
print(f"Response: {resp.json()}")

## 2. Single Prediction

In [ ]:
features = [round(random.uniform(0, 1), 4) for _ in range(NUM_FEATURES)]

resp = httpx.post(
    f"{API_URL}/predict",
    json={"model_id": MODEL_ID, "features": features},
)
print(f"Status: {resp.status_code}")
result = resp.json()
print(f"Prediction: {result.get('result')}")
print(f"Confidence: {result.get('confidence', {}).get('value', 'N/A')}")
print(f"Latency: {result.get('latency_ms', 'N/A')} ms")
print(f"Model Version: {result.get('version', 'N/A')}")

## 3. Batch Latency Benchmark

In [ ]:
NUM_REQUESTS = 100
latencies = []

for _ in range(NUM_REQUESTS):
    features = [round(random.uniform(0, 1), 4) for _ in range(NUM_FEATURES)]
    start = time.perf_counter()
    resp = httpx.post(
        f"{API_URL}/predict",
        json={"model_id": MODEL_ID, "features": features},
    )
    elapsed_ms = (time.perf_counter() - start) * 1000
    if resp.is_success:
        latencies.append(elapsed_ms)

print(f"Requests: {NUM_REQUESTS} | Successful: {len(latencies)}")
if latencies:
    latencies.sort()
    n = len(latencies)
    print(f"p50 latency: {latencies[n // 2]:.1f} ms")
    print(f"p95 latency: {latencies[int(n * 0.95)]:.1f} ms")
    print(f"p99 latency: {latencies[int(n * 0.99)]:.1f} ms")
    print(f"Mean: {statistics.mean(latencies):.1f} ms")
    print(f"Min: {min(latencies):.1f} ms | Max: {max(latencies):.1f} ms")
else:
    print("No successful requests — check API status")

## 4. Model Registry Info

In [ ]:
resp = httpx.get(f"{API_URL}/models/{MODEL_ID}")
if resp.is_success:
    model = resp.json()
    print(f"Model ID: {model['model_id']}")
    print(f"Version:  {model['version']}")
    print(f"Status:   {model['status']}")
    print(f"Metadata: {model.get('metadata', {})}")
else:
    print(f"Error: {resp.status_code} — {resp.text}")